#Cam0 정면데이터 출력본

In [4]:
import os
import io
from functools import lru_cache

import pandas as pd
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
from openpyxl import Workbook, load_workbook

# ==========================================
# 1. 경로 및 설정
# ==========================================
ROOT_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/2작기"
SAVE_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/2작기/bed_timeseries_review_topcam1.xlsx"

EXCEL_COLUMNS = ["image_info", "bednum", "date", "hhmm", "camera", "특이사항"]
THUMBNAIL_SIZE = (300, 300)
TARGET_CAM = "cam0"  # 카메라 고정 설정

# 전역 변수
raw_all_df = pd.DataFrame()
filtered_df = pd.DataFrame()
current_bed_no = ""
bed_list = []

# ==========================================
# 2. 엑셀 관련
# ==========================================
def ensure_excel_file():
    if os.path.exists(SAVE_PATH):
        return

    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    wb = Workbook()
    ws = wb.active
    ws.title = "review"
    ws.append(EXCEL_COLUMNS)
    wb.save(SAVE_PATH)

def append_rows_to_excel(rows):
    ensure_excel_file()
    wb = load_workbook(SAVE_PATH)
    ws = wb["review"]

    for row in rows:
        ws.append([row[col] for col in EXCEL_COLUMNS])

    wb.save(SAVE_PATH)
    wb.close()

def get_last_processed_bed():
    if not os.path.exists(SAVE_PATH):
        return None

    try:
        wb = load_workbook(SAVE_PATH, read_only=True, data_only=True)
        ws = wb["review"]

        last_bed = None
        first = True
        for row in ws.iter_rows(values_only=True):
            if first:
                first = False
                continue
            if row and len(row) > 1 and row[1]:
                last_bed = str(row[1])

        wb.close()
        return last_bed
    except Exception:
        return None

# ==========================================
# 3. 전수 스캔 및 필터링
# ==========================================
def scan_all_files(root):
    print(f"[{os.path.basename(root)}] 전수 스캔 중...")

    data = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.lower().endswith(".jpg"):
                continue

            try:
                # bed03_20260407_081723_cam0.jpg
                stem = fname[:-4]
                p = stem.split("_")
                if len(p) != 4:
                    continue

                bednum, date, hhmmss, cam = p
                full_path = os.path.join(dirpath, fname)

                data.append({
                    "bednum": bednum,
                    "date": date,
                    "hhmmss": hhmmss,
                    "hhmm": hhmmss[:4],
                    "cam": cam,
                    "path": full_path
                })
            except Exception:
                continue

    df = pd.DataFrame(data)
    if not df.empty:
        df = df.sort_values(["bednum", "date", "hhmmss"]).reset_index(drop=True)
    return df

def update_camera_filter():
    global filtered_df, bed_list

    cam_df = raw_all_df[raw_all_df["cam"] == TARGET_CAM].copy()

    if cam_df.empty:
        filtered_df = pd.DataFrame()
        bed_list = []
        with out:
            clear_output()
            print(f"경고: {TARGET_CAM} 이미지가 없습니다.")
        return

    filtered_df = cam_df.loc[cam_df.groupby(["bednum", "date"])["hhmmss"].idxmin()].copy()
    filtered_df = filtered_df.sort_values(["bednum", "date", "hhmmss"]).reset_index(drop=True)
    bed_list = sorted(filtered_df["bednum"].unique())

    status_html.value = f"<b>{TARGET_CAM}</b> 기준 {len(bed_list)}개 베드 로드 완료"

    if current_bed_no and current_bed_no in bed_list:
        render_bed_grid(current_bed_no)
    elif bed_list:
        render_bed_grid(bed_list[0])

# ==========================================
# 4. 이미지 로딩 최적화
# ==========================================
@lru_cache(maxsize=512)
def load_thumbnail_bytes(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        img.thumbnail(THUMBNAIL_SIZE)

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return buffer.getvalue()

def make_image_card(path, date_text, time_text):
    image_widget = widgets.Image(
        value=load_thumbnail_bytes(path),
        format="jpeg",
        width=300,
        height=300
    )

    caption = widgets.HTML(
        value=f"<div style='text-align:center; font-size:14px;'>{date_text} {time_text}</div>"
    )

    return widgets.VBox(
        [image_widget, caption],
        layout=widgets.Layout(
            width="320px",
            align_items="center",
            border="1px solid #ddd",
            padding="8px"
        )
    )

# ==========================================
# 5. 렌더링
# ==========================================
def render_bed_grid(bed_id):
    global current_bed_no

    if bed_id not in bed_list:
        status_html.value = f"<span style='color:red;'>없는 베드입니다: {bed_id}</span>"
        return

    current_bed_no = bed_id
    bed_data = filtered_df[filtered_df["bednum"] == bed_id].copy()
    bed_data = bed_data.sort_values(["date", "hhmmss"])

    with out:
        clear_output(wait=True)
        print(f"[{TARGET_CAM}] 모니터링: {bed_id} ({len(bed_data)} days)")

        cards = []
        for row in bed_data.to_dict("records"):
            cards.append(make_image_card(row["path"], row["date"], row["hhmm"]))

        rows = []
        for i in range(0, len(cards), 3):
            rows.append(
                widgets.HBox(
                    cards[i:i+3],
                    layout=widgets.Layout(gap="10px")
                )
            )

        display(widgets.VBox(rows, layout=widgets.Layout(gap="10px")))

    note_input.value = ""

# ==========================================
# 6. 저장 후 다음 베드
# ==========================================
def save_and_next(_=None):
    global current_bed_no

    if not current_bed_no:
        status_html.value = "<span style='color:red;'>현재 베드가 없습니다.</span>"
        return

    bed_data = filtered_df[filtered_df["bednum"] == current_bed_no].copy()
    bed_data = bed_data.sort_values(["date", "hhmmss"])

    if bed_data.empty:
        status_html.value = "<span style='color:red;'>저장할 데이터가 없습니다.</span>"
        return

    save_button.disabled = True
    save_button.description = "저장 중..."

    try:
        new_rows = []
        for row in bed_data.to_dict("records"):
            new_rows.append({
                "image_info": f"{row['bednum']}_{row['date']}_{row['hhmm']}_{row['cam']}",
                "bednum": row["bednum"],
                "date": row["date"],
                "hhmm": row["hhmm"],
                "camera": row["cam"],
                "특이사항": note_input.value
            })

        append_rows_to_excel(new_rows)

        idx = bed_list.index(current_bed_no)
        if idx < len(bed_list) - 1:
            next_bed = bed_list[idx + 1]
            status_html.value = f"<span style='color:green;'>{current_bed_no} 저장 완료 -> {next_bed}로 이동</span>"
            render_bed_grid(next_bed)
        else:
            status_html.value = f"<span style='color:green;'>{current_bed_no} 저장 완료. 모든 검사가 끝났습니다.</span>"
            with out:
                clear_output(wait=True)
                print("모든 검사가 완료되었습니다!")

    finally:
        save_button.disabled = False
        save_button.description = "저장 후 입력"

# ==========================================
# 7. 검색
# ==========================================
def search_bed(_=None):
    bed_id = search_box.value.strip()
    if not bed_id:
        return

    render_bed_grid(bed_id)

# ==========================================
# 8. UI
# ==========================================
ensure_excel_file()
raw_all_df = scan_all_files(ROOT_PATH)

search_box = widgets.Text(
    placeholder="bed01",
    description="베드검색:"
)

note_input = widgets.Text(
    placeholder="메모 입력 후 버튼 클릭",
    description="비고:",
    layout=widgets.Layout(width="500px")
)

save_button = widgets.Button(
    description="저장 후 입력",
    button_style="success",
    layout=widgets.Layout(width="140px")
)

status_html = widgets.HTML()
out = widgets.Output()

search_box.on_submit(search_bed)
save_button.on_click(save_and_next)

display(
    widgets.HBox([search_box]),
    out,
    widgets.HBox([note_input, save_button]),
    status_html
)

update_camera_filter()

last_processed = get_last_processed_bed()
if last_processed in bed_list:
    idx = bed_list.index(last_processed)
    next_idx = min(idx + 1, len(bed_list) - 1)
    render_bed_grid(bed_list[next_idx])
elif bed_list:
    render_bed_grid(bed_list[0])
else:
    status_html.value = "<span style='color:red;'>표시할 베드가 없습니다.</span>"


[2작기] 전수 스캔 중...


Output()

HTML(value='')